# Steal decision demo: one function, one table

This walks through `src/predict.py`'s `predict_steal_decision` -- the
single entry point built for the website handoff. It wraps the two
existing decision layers (RE24 for most of the game, win-probability
break-even for high-leverage late/close situations -- see
`win_probability.is_high_leverage`) so that a caller gets one row back with
every number the UI should show, not just a GO/HOLD verdict:

| column | meaning |
|---|---|
| `win_prob_current` | win probability right now, before the steal (always shown, whichever layer makes the decision) |
| `win_prob_if_success` | win probability if the steal succeeds |
| `break_even` | minimum success rate needed to justify the attempt, in the SAME units as `p_success` regardless of which layer produced it |
| `p_success` | the trained model's predicted success probability, from runner/pitcher/catcher inputs |
| `decision` | `GO` if `p_success > break_even`, else `HOLD` |

No new decision math lives here -- this notebook just exercises the
function the same way a website backend would: build the tables and fit
the model once, then call `predict_steal_decision` per situation.

In [1]:
import sys
sys.path.insert(0, "..")

from src.predict import load_tables, load_model, predict_steal_decision, predict_steal_decisions_table

print("Building RE24 + win-probability tables (reads a decade of play-by-play once)...")
tables = load_tables(base_dir="../data")
print("Fitting the model...")
model, medians = load_model(features_path="../data/sample/features_2023_2025.csv")
print("done")

Building RE24 + win-probability tables (reads a decade of play-by-play once)...


Fitting the model...


  (early stopping on a validation slice picked 234 rounds; refitting on the full training set with that round count)
done


## One situation at a time

`predict_steal_decision` takes game state (inning, half, outs, base state,
score, target base) plus player inputs (sprint speed, pop time, prior
rates, ...) and returns a dict with everything above.

In [2]:
row = predict_steal_decision(
    tables, model, medians,
    inning=9, half=1, outs=2, base_code="1__", score_diff=-1, target="2",
    runner_sprint_speed=29.5, catcher_pop_time=1.95,
    runner_prior_sr=0.82, runner_prior_att=25,
)
row

{'inning': 9,
 'half': 1,
 'outs': 2,
 'base_code': '1__',
 'score_diff': -1,
 'target': '2',
 'win_prob_current': 0.07123655913978495,
 'win_prob_if_success': 0.15333333333333332,
 'break_even': 0.4645862552594671,
 'p_success': 0.7954053282737732,
 'decision': 'GO',
 'layer': 'WP',
 'min_n': 150,
 'low_confidence': False,
 'sources': ('exact', 'exact', 'certain (home team trailing, game over)')}

Down 1, bottom of the 9th, 2 outs, runner on 1st: win probability right
now is only ~7%, so there's not much to protect -- break-even drops well
below RE24's early-game baseline (see `README.md`, "Win probability for
late/close games"), and a runner with an 82% prior success rate clears
that bar easily -- `GO`.

## Many situations at once -- one table, decision as the final column

`predict_steal_decisions_table` takes a list of situations and returns a
single `pandas.DataFrame`, exactly the shape a website table would want.

In [3]:
situations = [
    # early game, tied -- RE24 regime
    dict(inning=3, half=0, outs=1, base_code="1__", score_diff=0, target="2",
         runner_sprint_speed=29.0, catcher_pop_time=2.0, runner_prior_sr=0.78, runner_prior_att=40),
    # down 1, bottom 9th, 2 outs -- high leverage, win-probability regime
    dict(inning=9, half=1, outs=2, base_code="1__", score_diff=-1, target="2",
         runner_sprint_speed=29.5, catcher_pop_time=1.95, runner_prior_sr=0.82, runner_prior_att=25),
    # tied, bottom 9th, 2 outs -- same base-out state, different score
    dict(inning=9, half=1, outs=2, base_code="1__", score_diff=0, target="2",
         runner_sprint_speed=29.5, catcher_pop_time=1.95, runner_prior_sr=0.82, runner_prior_att=25),
    # leading by 1, late -- protect the lead
    dict(inning=8, half=1, outs=1, base_code="1__", score_diff=1, target="2",
         runner_sprint_speed=27.0, catcher_pop_time=2.0, runner_prior_sr=0.70, runner_prior_att=15),
    # early game, steal of third
    dict(inning=5, half=0, outs=0, base_code="_2_", score_diff=2, target="3",
         runner_sprint_speed=30.0, catcher_pop_time=1.9, runner_prior_sr=0.85, runner_prior_att=20),
]
df = predict_steal_decisions_table(tables, model, medians, situations)
df.style.format({
    "win_prob_current": "{:.1%}", "win_prob_if_success": "{:.1%}",
    "break_even": "{:.1%}", "p_success": "{:.1%}",
})

,inning,half,outs,base_code,score_diff,target,win_prob_current,win_prob_if_success,break_even,p_success,decision,layer,min_n,low_confidence
0,3,0,1,1__,0,2,47.7%,50.7%,73.7%,76.9%,GO,RE24,286,False
1,9,1,2,1__,-1,2,7.1%,15.3%,46.5%,79.5%,GO,WP,150,False
2,9,1,2,1__,0,2,58.0%,61.7%,64.8%,76.1%,GO,WP,180,False
3,8,1,1,1__,1,2,90.1%,87.9%,100.0%,69.1%,HOLD,WP,124,False
4,5,0,0,_2_,2,3,84.2%,83.3%,76.3%,75.8%,HOLD,RE24,24,False


Reading across: the `layer` column shows which decision engine actually
answered -- `RE24` early in the game, `WP` once `is_high_leverage` trips
(7th inning or later, score within 3). The leading-by-1 row's break-even
hits 100% -- win-probability found nothing to gain and a lot to lose, so
it's a certain `HOLD` regardless of the runner's skill. `min_n` /
`low_confidence` flag when an answer is backed by a thin historical
sample -- worth surfacing in a UI rather than presenting every number with
the same false precision.

## Two things this function handles gracefully, not just the happy path

**1. Missing player inputs.** A website user might not know (or the app
might not have) a specific runner's sprint speed, age, or a catcher's pop
time. Leaving those `None` falls back to the training-set median with a
`_missing` flag, the same way `src/features.py` already handles an
unmatched Statcast join -- no crash, no nonsensical zero.

In [4]:
row_missing = predict_steal_decision(
    tables, model, medians,
    inning=9, half=1, outs=2, base_code="1__", score_diff=-1, target="2",
    runner_sprint_speed=None, runner_age=None, catcher_pop_time=None,
    runner_prior_sr=0.82, runner_prior_att=25,
)
row_missing["p_success"], row_missing["decision"]

(0.7723446488380432, 'GO')

**2. A base/out combo RE24 has no data for.** `should_steal`/
`break_even_rate` do a bare dictionary lookup that would raise `KeyError`
in that case. `predict_steal_decision` catches it and falls through to the
win-probability path instead, which already has a graduated fallback chain
(`win_prob_lookup`: widen the score window, then average base codes, then
flag "no data" as a last resort) -- a website shouldn't 500 because a user
picked an exotic base-out state.

In [5]:
gapped_tables = dict(tables, re24={})  # simulate RE24 having zero coverage
row_gap = predict_steal_decision(
    gapped_tables, model, medians,
    inning=3, half=0, outs=1, base_code="1__", score_diff=0, target="2",
    runner_sprint_speed=29.0, catcher_pop_time=2.0, runner_prior_sr=0.78, runner_prior_att=40,
)
row_gap["layer"], row_gap["break_even"]

('WP (RE24 had no data)', 0.5785199169745525)

No exception -- the `layer` column flags exactly what happened
(`"WP (RE24 had no data)"`) instead of silently guessing or crashing.